# 02 · Incidents: Cleaning

Now we **fix** every problem we found in notebook 01, step by step, and save two clean tables:

| Output file | One row per | Used for |
|---|---|---|
| `data/interim/incident_events_clean.parquet` | update (event) | process analysis (e.g. reassignment paths) |
| `data/interim/incidents_clean.parquet` | **ticket** | feature engineering (notebook 03) and root-cause analysis |

We also write a **data quality report** (`reports/incidents_quality_report.csv`) that records how many rows each step removed and why. Judges love this.

**Why `.parquet` and not `.csv`?** Parquet files keep the column types (dates stay dates, numbers stay numbers), are ~10× smaller, and load much faster.

In [1]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd()
while not (ROOT / "requirements.txt").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RAW = ROOT / "data/raw/incidents"
INTERIM = ROOT / "data/interim"
REPORTS = ROOT / "reports"
INTERIM.mkdir(parents=True, exist_ok=True)
pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 200)

# The quality log: every cleaning step adds one line here
quality_log = []
def log(step, rows_before, rows_after, note=""):
    quality_log.append({"step": step, "rows_before": rows_before, "rows_after": rows_after,
                        "rows_removed": rows_before - rows_after, "note": note})
    print(f"{step:45s} {rows_before:>8,} → {rows_after:>8,}  (removed {rows_before - rows_after:,})  {note}")

## Step 1: Load, treating `?` as missing  *(fixes finding #2)*

In [2]:
df = pd.read_csv(RAW / "incident_event_log.csv", na_values="?", low_memory=False)
start_rows = len(df)
log("1. load raw file", start_rows, len(df), f"{df['number'].nunique():,} tickets")

# Missing % per column BEFORE cleaning (saved for the report)
missing_before = df.isna().mean().mul(100).round(2)

1. load raw file                               141,712 →  141,712  (removed 0)  24,918 tickets


## Step 2: Drop rows with the broken state `-100`  *(fixes finding #4)*

In [3]:
before = len(df)
df = df[df["incident_state"] != "-100"].copy()
log("2. drop incident_state == -100", before, len(df))

2. drop incident_state == -100                 141,712 →  141,707  (removed 5)  


## Step 3: Remove exact duplicate rows
A duplicate is a row where **every** column is identical to another row. There may be none, but we always check.

In [4]:
before = len(df)
df = df.drop_duplicates()
log("3. drop exact duplicate rows", before, len(df))

3. drop exact duplicate rows                   141,707 →  141,707  (removed 0)  


## Step 4: Convert date text into real dates  *(fixes finding #5)*
`errors="coerce"` means: if a value can't be read as a date, make it missing (`NaT`) instead of crashing.

In [5]:
date_cols = ["opened_at", "sys_created_at", "sys_updated_at", "resolved_at", "closed_at"]
for c in date_cols:
    df[c] = pd.to_datetime(df[c], format="%d/%m/%Y %H:%M", errors="coerce")
df[date_cols].dtypes

opened_at         datetime64[ns]
sys_created_at    datetime64[ns]
sys_updated_at    datetime64[ns]
resolved_at       datetime64[ns]
closed_at         datetime64[ns]
dtype: object

## Step 5: Turn `"2 - Medium"` into the number `2`  *(fixes finding #6)*
`.str.extract(r"(\d+)")` pulls out the first number in the text. (`r"(\d+)"` is a *regular expression* meaning "one or more digits".)
Remember: **1 = most severe**.

In [6]:
for c in ["impact", "urgency", "priority"]:
    df[f"{c}_n"] = df[c].str.extract(r"(\d+)", expand=False).astype(float)
df[["impact", "impact_n", "urgency", "urgency_n", "priority", "priority_n"]].drop_duplicates().head(8)

,impact,impact_n,urgency,urgency_n,priority,priority_n
0,2 - Medium,2.0,2 - Medium,2.0,3 - Moderate,3.0
25,1 - High,1.0,2 - Medium,2.0,2 - High,2.0
127,3 - Low,3.0,3 - Low,3.0,4 - Low,4.0
158,2 - Medium,2.0,3 - Low,3.0,4 - Low,4.0
481,3 - Low,3.0,2 - Medium,2.0,4 - Low,4.0
688,1 - High,1.0,1 - High,1.0,1 - Critical,1.0
1514,3 - Low,3.0,1 - High,1.0,3 - Moderate,3.0
1904,2 - Medium,2.0,1 - High,1.0,2 - High,2.0


`notify` only has two values, so we turn it into True/False.

In [7]:
df["notify_email"] = df["notify"].eq("Send Email")

## Step 6: Sort events into time order
Within each ticket, `sys_mod_count` counts updates 0, 1, 2, ... so sorting by it puts each ticket's history in order.

In [8]:
df = df.sort_values(["number", "sys_mod_count", "sys_updated_at"]).reset_index(drop=True)

## Step 7: Keep only the main time window  *(fixes finding #7)*
We keep tickets **opened** between 29 Feb and 31 May 2016 (`opened_at` is the same on every event of a ticket).

In [9]:
WINDOW_START, WINDOW_END = pd.Timestamp("2016-02-29"), pd.Timestamp("2016-06-01")
before, before_t = len(df), df["number"].nunique()
df = df[(df["opened_at"] >= WINDOW_START) & (df["opened_at"] < WINDOW_END)].copy()
log("7. keep tickets opened 29 Feb – 31 May 2016", before, len(df),
    f"tickets {before_t:,} → {df['number'].nunique():,}")

7. keep tickets opened 29 Feb – 31 May 2016    141,707 →  140,060  (removed 1,647)  tickets 24,918 → 24,644


## Step 8: Sanity checks on the timeline
A ticket can't be resolved or closed before it was opened. Let's count such cases.

In [10]:
bad_resolved = (df["resolved_at"] < df["opened_at"]).sum()
bad_closed = (df["closed_at"] < df["opened_at"]).sum()
print("Events where resolved_at < opened_at:", bad_resolved)
print("Events where closed_at  < opened_at:", bad_closed)

bad_tickets = df.loc[(df["resolved_at"] < df["opened_at"]) | (df["closed_at"] < df["opened_at"]), "number"].unique()
before = len(df)
df = df[~df["number"].isin(bad_tickets)].copy()
log("8. drop tickets with impossible timelines", before, len(df), f"{len(bad_tickets)} tickets")

Events where resolved_at < opened_at: 0
Events where closed_at  < opened_at: 0
8. drop tickets with impossible timelines      140,060 →  140,060  (removed 0)  0 tickets


## Step 9: Save the clean event log
We drop the 5 almost-empty columns here *(finding #3)*, but first remember whether each ticket had a value at all.

In [11]:
empty_cols = ["caused_by", "vendor", "cmdb_ci", "rfc", "problem_id"]
for c in empty_cols:
    df[f"has_{c}"] = df[c].notna()

events = df.drop(columns=empty_cols + ["notify"])
events.to_parquet(INTERIM / "incident_events_clean.parquet", index=False)
print("Saved incident_events_clean.parquet:", events.shape)

Saved incident_events_clean.parquet: (140060, 39)


## Step 10: Collapse to ONE ROW PER TICKET  *(fixes finding #1)*

This is the most important step. For each ticket we take:
- the **first** event → what we knew **when the ticket was opened** (safe to use as model features)
- the **last** event → how the ticket **ended** (the target, and outcome columns for root-cause analysis only)

We prefix outcome columns with `final_` so it's always obvious which columns are "from the future".

⚠ **Beginner trap:** `df.groupby("number").first()` does **not** give the first row! It gives the first *non-missing value of each column separately*,
which can quietly pull information from later events. We use `drop_duplicates(keep="first")` instead, which keeps the real first row.

In [12]:
first = events.drop_duplicates("number", keep="first").set_index("number")
last = events.drop_duplicates("number", keep="last").set_index("number")
g = events.groupby("number")

# What we knew at opening time (from the FIRST event)
at_open_cols = ["opened_at", "caller_id", "opened_by", "contact_type", "location", "category",
                "subcategory", "u_symptom", "impact_n", "urgency_n", "priority_n",
                "assignment_group", "knowledge", "u_priority_confirmation", "notify_email",
                "has_cmdb_ci", "has_problem_id", "has_rfc", "has_vendor", "has_caused_by",
                "incident_state"]
tickets = first[at_open_cols].rename(columns={"incident_state": "first_state_seen"})

# How it ended (from the LAST event) - outcomes, NOT features for the at-open model
tickets["final_made_sla"] = last["made_sla"]
tickets["final_reassignment_count"] = last["reassignment_count"]
tickets["final_reopen_count"] = last["reopen_count"]
tickets["final_sys_mod_count"] = last["sys_mod_count"]
tickets["final_assignment_group"] = last["assignment_group"]
tickets["final_closed_code"] = last["closed_code"]
tickets["resolved_at"] = last["resolved_at"]
tickets["closed_at"] = last["closed_at"]
tickets["n_events"] = g.size()

# THE TARGET: 1 = breached SLA, 0 = made SLA
tickets["breached"] = (~tickets["final_made_sla"]).astype(int)

# Handy outcome measure for analysis
tickets["hours_to_resolve"] = (tickets["resolved_at"] - tickets["opened_at"]).dt.total_seconds() / 3600

tickets = tickets.reset_index()
print("One row per ticket:", tickets.shape)
print(f"Breach rate: {tickets['breached'].mean():.1%}")
tickets.head()

One row per ticket: (24644, 33)
Breach rate: 36.6%


,number,opened_at,caller_id,opened_by,contact_type,location,category,subcategory,u_symptom,impact_n,urgency_n,priority_n,assignment_group,knowledge,u_priority_confirmation,notify_email,has_cmdb_ci,has_problem_id,has_rfc,has_vendor,has_caused_by,first_state_seen,final_made_sla,final_reassignment_count,final_reopen_count,final_sys_mod_count,final_assignment_group,final_closed_code,resolved_at,closed_at,n_events,breached,hours_to_resolve
0,INC0000045,2016-02-29 01:16:00,Caller 2403,Opened by 8,Phone,Location 143,Category 55,Subcategory 170,Symptom 72,2.0,2.0,3.0,Group 56,True,False,False,False,False,False,False,False,New,True,0,0,4,Group 56,code 5,2016-02-29 11:29:00,2016-03-05 12:00:00,4,0,10.216667
1,INC0000047,2016-02-29 04:40:00,Caller 2403,Opened by 397,Phone,Location 165,Category 40,Subcategory 215,Symptom 471,2.0,2.0,3.0,Group 70,True,False,False,False,False,False,False,False,New,True,1,0,8,Group 24,code 5,2016-03-01 09:52:00,2016-03-06 10:00:00,9,0,29.200000
2,INC0000057,2016-02-29 06:10:00,Caller 4416,Opened by 8,Phone,Location 204,Category 20,Subcategory 125,Symptom 471,2.0,2.0,3.0,Group 70,True,False,False,False,False,False,False,False,New,True,0,0,6,Group 70,code 10,2016-03-01 02:55:00,2016-03-06 03:00:00,7,0,20.750000
3,INC0000060,2016-02-29 06:38:00,Caller 4491,Opened by 180,Phone,Location 204,Category 9,Subcategory 97,Symptom 450,2.0,2.0,3.0,Group 25,True,False,False,False,False,False,False,False,New,True,0,0,3,Group 25,code 3,2016-03-02 12:06:00,2016-03-07 13:00:00,4,0,53.466667
4,INC0000062,2016-02-29 06:58:00,Caller 3765,Opened by 180,Phone,Location 93,Category 53,Subcategory 168,Symptom 232,2.0,2.0,3.0,Group 70,True,False,False,False,False,False,False,False,New,False,1,0,7,Group 23,code 7,2016-02-29 15:51:00,2016-03-05 16:00:00,8,1,8.883333


## Step 11: Fill missing text values with `"Unknown"`
For text (category-like) columns, a missing value can itself mean something (e.g. "no team assigned yet").
So instead of deleting those tickets, we label them `"Unknown"`. Models can then learn from it.

In [13]:
text_cols = ["caller_id", "opened_by", "contact_type", "location", "category", "subcategory",
             "u_symptom", "assignment_group", "final_assignment_group", "final_closed_code"]
print("Missing before fill:\n", tickets[text_cols].isna().sum().to_string())
tickets[text_cols] = tickets[text_cols].fillna("Unknown")

# Numbers: are any severity values missing?
print("\nMissing severity numbers:", tickets[["impact_n", "urgency_n", "priority_n"]].isna().sum().to_dict())

Missing before fill:
 caller_id                    3
opened_by                  714
contact_type                 0
location                    52
category                    26
subcategory                 56
u_symptom                 5674
assignment_group           906
final_assignment_group    2157
final_closed_code          105

Missing severity numbers: {'impact_n': 0, 'urgency_n': 0, 'priority_n': 0}


## Step 12: Final checks
- Each ticket number must appear **exactly once**.
- The target must have no missing values.

In [14]:
assert tickets["number"].is_unique, "Duplicate tickets found!"
assert tickets["breached"].notna().all(), "Missing target values!"
log("12. final: one row per ticket", len(events), len(tickets), "events collapsed into tickets")
print("\nAll checks passed ✔")

12. final: one row per ticket                  140,060 →   24,644  (removed 115,416)  events collapsed into tickets

All checks passed ✔


## Step 13: Save the clean ticket table and the quality report

In [15]:
tickets.to_parquet(INTERIM / "incidents_clean.parquet", index=False)
print("Saved incidents_clean.parquet:", tickets.shape)

report = pd.DataFrame(quality_log)
report.to_csv(REPORTS / "incidents_quality_report.csv", index=False)

missing_cols = pd.DataFrame({
    "missing_%_raw_events": missing_before,
}).sort_values("missing_%_raw_events", ascending=False)
missing_cols.to_csv(REPORTS / "incidents_missing_values_raw.csv")
report

Saved incidents_clean.parquet: (24644, 33)


,step,rows_before,rows_after,rows_removed,note
0,1. load raw file,141712,141712,0,"24,918 tickets"
1,2. drop incident_state == -100,141712,141707,5,
2,3. drop exact duplicate rows,141707,141707,0,
3,7. keep tickets opened 29 Feb – 31 May 2016,141707,140060,1647,"tickets 24,918 → 24,644"
4,8. drop tickets with impossible timelines,140060,140060,0,0 tickets
5,12. final: one row per ticket,140060,24644,115416,events collapsed into tickets


### What we have now
- ~24,600 tickets, one row each, with clear `open_...`/at-opening columns and `final_...` outcome columns
- target column `breached` (1 = breached SLA)
- a documented record of every row removed

➡ Next: **03_incidents_features.ipynb**, turning these columns into model features.